# 🎧 Banc de Test & Comparaison des Checkpoints Reverb-TTS (INT4 & INT8)

Ce notebook permet de :
1. **Lister automatiquement tous vos checkpoints** sauvegardés dans `Mon Drive/reverb_tts/narrateur_model/checkpoints_history/` (ainsi que `last`).
2. **Tester et comparer facilement plusieurs époques** en **INT4** (`group_size=32`, ~191 Mo) et en **INT8** (`per-channel`, ~307 Mo) sur un paragraphe dark fantasy de 4 phrases style *Shadow Slave*.
3. **Exporter définitivement** le meilleur checkpoint choisi sur votre Google Drive.

## 1. Installation & Connexion Google Drive

In [ ]:
!pip uninstall -y -q torchao
!pip install -q reverb-tts peft safetensors huggingface_hub soundfile

In [ ]:
import glob, json, os, shutil, zipfile
import torch
import soundfile as sf
import IPython.display as ipd
from google.colab import drive

if not os.path.exists("/content/drive"):
    drive.mount("/content/drive")

BASE_MODEL = "software-mansion/Reverb-TTS"
DATASET_ZIP = "/content/drive/MyDrive/reverb_tts/dataset_reverb_narrateur.zip"
DRIVE_MODEL_DIR = "/content/drive/MyDrive/reverb_tts/narrateur_model"
HISTORY_DIR = os.path.join(DRIVE_MODEL_DIR, "checkpoints_history")
LAST_DIR = os.path.join(DRIVE_MODEL_DIR, "last")
WORK_DIR = "/content/work_reverb_test"
os.makedirs(WORK_DIR, exist_ok=True)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"🖥️ Appareil utilisé pour le test : {DEVICE}")

## 2. Liste des Checkpoints disponibles & Extraction de la Voix de Référence

In [ ]:
# 1. Extraire ref_narrateur.wav et ref_narrateur.txt depuis dataset_reverb_narrateur.zip
REF_DIR = os.path.join(WORK_DIR, "ref")
os.makedirs(REF_DIR, exist_ok=True)
REF_WAV = os.path.join(REF_DIR, "ref_narrateur.wav")
REF_TXT_PATH = os.path.join(REF_DIR, "ref_narrateur.txt")

if not os.path.exists(REF_WAV):
    assert os.path.exists(DATASET_ZIP), f"Dataset ZIP introuvable : {DATASET_ZIP}"
    with zipfile.ZipFile(DATASET_ZIP, "r") as z:
        for member in z.namelist():
            if member.endswith("ref_narrateur.wav"):
                with z.open(member) as src, open(REF_WAV, "wb") as dst:
                    shutil.copyfileobj(src, dst)
            elif member.endswith("ref_narrateur.txt"):
                with z.open(member) as src, open(REF_TXT_PATH, "wb") as dst:
                    shutil.copyfileobj(src, dst)

REF_TEXT = open(REF_TXT_PATH, encoding="utf-8").read().strip()
print(f"🎙️ Référence chargée ({REF_WAV}) :\n   « {REF_TEXT} »\n")
ipd.display(ipd.Audio(REF_WAV))

# 2. Lister tous les checkpoints LoRA disponibles
AVAILABLE_CKPTS = []
if os.path.exists(HISTORY_DIR):
    for name in sorted(os.listdir(HISTORY_DIR)):
        p = os.path.join(HISTORY_DIR, name)
        if os.path.exists(os.path.join(p, "adapter_config.json")):
            AVAILABLE_CKPTS.append((name, p))
if os.path.exists(os.path.join(LAST_DIR, "adapter_config.json")):
    AVAILABLE_CKPTS.append(("last", LAST_DIR))

print(f"\n📂 {len(AVAILABLE_CKPTS)} checkpoint(s) disponible(s) :")
for idx, (name, path) in enumerate(AVAILABLE_CKPTS):
    state_p = os.path.join(path, "trainer_state.json")
    info = ""
    if os.path.exists(state_p):
        st = json.load(open(state_p))
        info = f" (epoch={st.get('epoch')}, loss={st.get('loss', 0):.4f})"
    print(f"  [{idx}] {name}{info}")

## 3. Sélection des Checkpoints et Test Comparatif **INT4** vs **INT8** (Paragraphe *Shadow Slave*)

In [ ]:
from safetensors.torch import save_file
from huggingface_hub import snapshot_download
from peft import PeftModel
from reverb_tts import TTS
from reverb_tts.model import QUANTIZED, SCALE

# === CHOISISSEZ LES CHECKPOINTS À TESTER ===
# Mettez une liste d'indices affichés ci-dessus (ex: [0, -1] pour comparer le 1er et le dernier,
# ou [-3, -2, -1] pour tester les 3 derniers checkpoints)
INDICES_A_TESTER = [-2, -1] if len(AVAILABLE_CKPTS) >= 2 else [-1]

# Variantes à tester : ("int4", "int8") ou seulement ("int4",)
VARIANTES_A_TESTER = ("int4", "int8")

# Paragraphe dark fantasy de 4 phrases inspiré de Shadow Slave
TEXTE_SHADOW_SLAVE = (
    "Sous le ciel mort du Royaume des Rêves, les ruines de la citadelle oubliée se dressaient comme les côtes brisées d'un léviathan ancien. "
    "Sunny retint son souffle tandis que son ombre glissait silencieusement sur les dalles de pierre glacée, attentive au moindre frémissement dans les ténèbres. "
    "Un grondement sourd résonna soudain au fond du gouffre, annonçant l'éveil d'une créature du Cauchemar que même les anciens dieux avaient crainte. "
    "Serrant la poignée de sa lame d'obsidienne, il sourit avec amertume face à l'abîme, sachant que personne ne viendrait le sauver."
)

# Téléchargement unique des fichiers de base fp32 et du tokenizer
src_root = snapshot_download(
    BASE_MODEL,
    allow_patterns=["model/safetensors/fp32/*", "model/tokenizer*", "model/chat_template.jinja"]
)
base_cfg = json.load(open(os.path.join(src_root, "model/safetensors/fp32/config.json"), "r", encoding="utf-8"))

def build_quantized_folder(merged_state, out_dir, variant_name):
    bits = 4 if variant_name == "int4" else 8
    group_size = 32 if bits == 4 else None
    model_dir = os.path.join(out_dir, "model")
    var_dir = os.path.join(model_dir, "safetensors", variant_name)
    os.makedirs(var_dir, exist_ok=True)

    for fn in os.listdir(os.path.join(src_root, "model")):
        src_f = os.path.join(src_root, "model", fn)
        if os.path.isfile(src_f):
            shutil.copy2(src_f, os.path.join(model_dir, fn))

    q_weights = {}
    for name, tensor in merged_state.items():
        if name.startswith(QUANTIZED) and tensor.ndim == 2 and name.endswith(".weight"):
            w = tensor.float()
            if bits == 4:
                wg = w.unflatten(-1, (-1, group_size))
                scale = (wg.abs().amax(dim=-1) / 7.0).clamp(min=1e-8).to(torch.float16)
                q = torch.clamp(torch.round(wg / scale.float()[..., None]), -8, 7).to(torch.int16) + 8
                q_flat = q.flatten(-2)
                packed = (q_flat[:, 0::2] | (q_flat[:, 1::2] << 4)).to(torch.uint8)
                q_weights[name] = packed
                q_weights[f"{name}{SCALE}"] = scale
            else:
                scale = (w.abs().amax(dim=-1, keepdim=True) / 127.0).clamp(min=1e-8).to(torch.float16)
                q = torch.clamp(torch.round(w / scale.float()), -127, 127).to(torch.int8)
                q_weights[name] = q
                q_weights[f"{name}{SCALE}"] = scale
        else:
            q_weights[name] = tensor.to(torch.float16) if tensor.is_floating_point() else tensor

    cfg = dict(base_cfg)
    cfg["quantization"] = {"bits": 4, "group_size": 32} if bits == 4 else {"bits": 8}
    with open(os.path.join(var_dir, "config.json"), "w", encoding="utf-8") as f:
        json.dump(cfg, f, indent=2)
    save_file(q_weights, os.path.join(var_dir, "model.safetensors"))
    return out_dir

print(f"📖 Texte de test (Shadow Slave, 4 phrases) :\n« {TEXTE_SHADOW_SLAVE} »\n")

for idx in INDICES_A_TESTER:
    ckpt_name, ckpt_path = AVAILABLE_CKPTS[idx]
    print("=" * 70)
    print(f"🔄 Fusion LoRA pour [{idx}] {ckpt_name}...")

    base_tts = TTS.from_pretrained(BASE_MODEL, variant="fp32", device="cpu")
    peft_llm = PeftModel.from_pretrained(base_tts.network.llm, ckpt_path)
    base_tts.network.llm = peft_llm.merge_and_unload()
    merged_state = {k: v.detach().cpu() for k, v in base_tts.network.state_dict().items()}
    del base_tts, peft_llm

    ckpt_work = os.path.join(WORK_DIR, f"build_{ckpt_name}")
    for variant in VARIANTES_A_TESTER:
        v_dir = build_quantized_folder(merged_state, os.path.join(ckpt_work, variant), variant)
        t = TTS.from_pretrained(v_dir, variant=variant, device=DEVICE)
        v = t.voice(REF_WAV, REF_TEXT, language="fr", prepare=False)
        audio = t.generate(TEXTE_SHADOW_SLAVE, v, language="fr", seed=42)
        wav_out = os.path.join(WORK_DIR, f"{ckpt_name}_{variant}.wav")
        sf.write(wav_out, audio, t.sample_rate)

        size_mb = os.path.getsize(os.path.join(v_dir, "model", "safetensors", variant, "model.safetensors")) / 1e6
        print(f"\n🎧 [{idx}] {ckpt_name} — {variant.upper()} ({size_mb:.1f} Mo) :")
        ipd.display(ipd.Audio(wav_out))
        del t
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

## 4. Sauvegarder votre Meilleur Checkpoint (INT4 + INT8) sur Google Drive
Une fois votre époque préférée identifiée à l'écoute ci-dessus, indiquez son indice `BEST_INDEX` pour copier ses modèles `export_int4` et `export_int8` dans `Mon Drive/reverb_tts/narrateur_model/`.

In [ ]:
BEST_INDEX = -1  # Remplacez par l'indice [idx] du meilleur checkpoint écouté ci-dessus

best_name, best_path = AVAILABLE_CKPTS[BEST_INDEX]
ckpt_work = os.path.join(WORK_DIR, f"build_{best_name}")

for variant in VARIANTES_A_TESTER:
    src_v = os.path.join(ckpt_work, variant)
    dst_v = os.path.join(DRIVE_MODEL_DIR, f"export_{variant}_{best_name}")
    if os.path.exists(dst_v):
        shutil.rmtree(dst_v)
    shutil.copytree(src_v, dst_v)
    print(f"✅ Sauvegardé sur Google Drive : {dst_v}")